# Thunder and Pacers Shot Data (Finals)

In [1]:
from nba_api.stats.endpoints import shotchartdetail
from nba_api.stats.static import players, teams
from nba_api.stats.endpoints import commonplayerinfo
from nba_api.stats.static import players
import pandas as pd

In [2]:
import time, random
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

def nba_session():
    s = requests.Session()

    # These headers matter for stats.nba.com
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Origin": "https://www.nba.com",
        "Referer": "https://www.nba.com/",
        "Connection": "keep-alive",
    })

    retry = Retry(
        total=8,
        connect=8,
        read=8,
        backoff_factor=1.2,              # exponential-ish backoff
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST"],
        raise_on_status=False,
        respect_retry_after_header=True,
    )

    adapter = HTTPAdapter(max_retries=retry, pool_connections=50, pool_maxsize=50)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s

SESSION = nba_session()

def polite_sleep(min_s=0.6, max_s=1.6):
    time.sleep(random.uniform(min_s, max_s))


In [3]:
import pandas as pd
from nba_api.stats.endpoints import leaguegamefinder

def get_finals_game_ids_for_player(player_id: int, season: str) -> list[str]:
    """
    Returns Finals game IDs for a player in a given season (e.g., '2016-17').
    Finals games have GAME_ID containing '040' in the series code.
    """
    gf = leaguegamefinder.LeagueGameFinder(
        player_id_nullable=player_id,
        season_nullable=season,
        season_type_nullable="Playoffs"
    )
    games = gf.get_data_frames()[0]  # game log

    # Finals filter: Finals series games have '040' in the GAME_ID (e.g., 0041600401)
    games["GAME_ID"] = games["GAME_ID"].astype(str)
    finals_games = games[games["GAME_ID"].str.contains("040")].copy()

    return finals_games["GAME_ID"].tolist()

In [4]:
from nba_api.stats.endpoints import shotchartdetail

def get_player_shots_for_game(player_id: int, season: str, game_id: str) -> pd.DataFrame:
    scd = shotchartdetail.ShotChartDetail(
        team_id=0,
        player_id=player_id,
        season_nullable=season,
        season_type_all_star="Playoffs",
        game_id_nullable=game_id,
        context_measure_simple="FGA"
    )
    df = scd.get_data_frames()[0]
    return df

In [5]:
import time

def finals_shots_last_10_seasons(player_name: str, start_year=2024, end_year=2025):
    """
    Pulls ALL shots taken by player in ALL NBA Finals games from seasons start_year-end_year.
    Seasons are strings like '2016-17'. end_year refers to the ending year of the season.
    """
    from nba_api.stats.static import players

    # find player_id
    p = players.find_players_by_full_name(player_name)[0]
    player_id = p["id"]

    all_shots = []
    summary = []

    for end in range(start_year + 1, end_year + 1):
        season = f"{end-1}-{str(end)[-2:]}"

        finals_game_ids = get_finals_game_ids_for_player(player_id, season)
        summary.append({"season": season, "finals_games_found": len(finals_game_ids)})

        for gid in finals_game_ids:
            try:
                df_game = get_player_shots_for_game(player_id, season, gid)
                if not df_game.empty:
                    all_shots.append(df_game)
            except Exception as e:
                print(f"Failed game {gid}: {e}")

            time.sleep(0.8)

    shots_df = pd.concat(all_shots, ignore_index=True) if all_shots else pd.DataFrame()
    summary_df = pd.DataFrame(summary)

    return shots_df, summary_df


In [6]:
shai_df, shai_summary = finals_shots_last_10_seasons(
    "Shai Gilgeous-Alexander",
    2024,
    2025
)

print("SHAI:", len(shai_df))

SHAI: 158


In [7]:
chet_df, chet_summary = finals_shots_last_10_seasons(
    "Chet Holmgren",
    2024,
    2025
)

print("CHET:", len(chet_df))

CHET: 76


In [8]:
combined_df = pd.concat([shai_df, chet_df], ignore_index=True)

print("TOTAL ROWS:", len(combined_df))
combined_df.head()

TOTAL ROWS: 234


,GRID_TYPE,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD,MINUTES_REMAINING,SECONDS_REMAINING,...,SHOT_ZONE_AREA,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM
0,Shot Chart Detail,0042400407,15,1628983,Shai Gilgeous-Alexander,1610612760,Oklahoma City Thunder,1,10,57,...,Center(C),8-16 ft.,14,70,132,1,1,20250622,OKC,IND
1,Shot Chart Detail,0042400407,18,1628983,Shai Gilgeous-Alexander,1610612760,Oklahoma City Thunder,1,10,17,...,Left Side(L),16-24 ft.,21,-185,107,1,0,20250622,OKC,IND
2,Shot Chart Detail,0042400407,29,1628983,Shai Gilgeous-Alexander,1610612760,Oklahoma City Thunder,1,9,35,...,Left Side(L),8-16 ft.,15,-150,-1,1,1,20250622,OKC,IND
3,Shot Chart Detail,0042400407,109,1628983,Shai Gilgeous-Alexander,1610612760,Oklahoma City Thunder,1,3,52,...,Right Side(R),8-16 ft.,12,125,7,1,0,20250622,OKC,IND
4,Shot Chart Detail,0042400407,115,1628983,Shai Gilgeous-Alexander,1610612760,Oklahoma City Thunder,1,3,18,...,Center(C),Less Than 8 ft.,6,6,63,1,1,20250622,OKC,IND


In [9]:
combined_df.to_csv(
    "Thunder Pacers Data/Thunder_Finals_Shots_2025.csv",
    index=False
)

print("Saved Thunder_Finals_Shots_2025.csv")

OSError: Cannot save file into a non-existent directory: 'Thunder Pacers Data'